# ED-MRDT v1.1 — PIP3-PTEN Full Analysis

Complete pipeline for the **phosphoinositide wave** system:

**PIP2 ↔ PIP3 conversion** driven by PI3K (kinase) and PTEN (phosphatase), with:
- Catalytic feedback (Hill-type autoactivation)
- Compartment exchange (cytosol ↔ membrane)
- PH-Akt biosensor (intramolecular FRET reporter for PIP3)
- Fire-diffuse-fire wave dynamics

Pipeline: **BD → Virtual FLIM → Image Analysis → CTE v2 (Causal Inference)**

---

## 1. Setup

In [ ]:
# === OPTION 1: Local execution (project already on disk) ===
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# === OPTION 2: Google Colab (uncomment these lines) ===
# !pip install numba scipy matplotlib pillow -q
# # Upload ed_mrdt.zip or clone:
# # !git clone https://github.com/<user>/ed_mrdt_v1.1.git
# # !unzip ed_mrdt.zip
# import sys; sys.path.insert(0, 'ed_mrdt_v1.1_PIP3-PTEN_PAPER_1')

import copy, time, json
import numpy as np
from IPython.display import display, Image, HTML

from ed_mrdt.config import load_config
from ed_mrdt.pipeline import SimulationPipeline
from ed_mrdt.video import generate_all_replica_videos
from l6_te_redesign import (
    sim_results_to_trials_dict,
    CausalPairSpecV2,
    compute_causal_flim_generic_v2,
    discretize_uniform,
    ensemble_conditional_te_discrete,
)

print('All imports OK')

## 2. Configuration (EDITABLE)

### 2a. Simulation Parameters

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT CONFIGURATION — EDIT FREELY                     ║
# ╚══════════════════════════════════════════════════════════════╝

YAML_CONFIG      = '../ed_mrdt/examples/pip3_pten_flim.yaml'
N_REPLICAS       = 5           # Independent runs (WARNING: each takes ~10 min)
N_FRAMES         = 10          # Frames per replica
N_BD_PER_FRAME   = 1000        # BD steps per frame (1000 x 2ms = 2s/frame)
BD_TIMESTEP      = 2e-3        # 2 ms (large timestep, harmonic potential)

# --- Transfer Entropy ---
N_SURROGATES     = 200
TE_N_BINS        = 2
TE_CANDIDATE_LAGS = (1, 2, 5)
TE_LAG_CRITERION = 'asymmetry'
TE_DETREND       = 'none'       # 'none', 'diff1', 'diff2'

# --- Video ---
VIDEO_FPS        = 3
VIDEO_DPI        = 120
VIDEO_FORMAT     = 'gif'

# --- FLIM ---
TAU_D_NS         = 4.1         # mTFP1 unquenched lifetime
DONOR_CHANNEL    = 0

# --- Output ---
OUTPUT_DIR       = '../results_PIP3_PTEN'
VIDEO_DIR        = f'{OUTPUT_DIR}/videos'

print(f'Estimated time: ~{N_REPLICAS * 10:.0f} minutes for {N_REPLICAS} replicas')
print(f'Effective TE points: {N_REPLICAS} trials x {N_FRAMES-1} = {N_REPLICAS*(N_FRAMES-1)}')

### 2b. PIP3-PTEN Biological Parameters

These override the YAML defaults for the phosphoinositide wave system.
Every parameter is documented with its biological reference.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  BIOLOGICAL PARAMETERS — PIP3/PTEN WAVE SYSTEM              ║
# ║  Ref: Matsuoka & Ueda (2018) Nat Commun 9:4481              ║
# ║       Arai et al (2010) PNAS 107:12399                      ║
# ║       Goryachev & Leda (2017) Mol Biol Cell 28:370          ║
# ╚══════════════════════════════════════════════════════════════╝

# --- Membrane ---
MEMBRANE_SIZE    = [10.0, 10.0]  # um x um (10x10 = 100 um2, fits ~1.6 wavelengths)
PIXELS           = 32            # 32x32 pixels -> 312 nm/pixel
PIXEL_SIZE_NM    = 312.5         # nm (= 10um / 32 pixels)
DWELL_TIME       = 5.0e-3       # 5 ms dwell time per pixel

# --- Populations (per um2 density -> total count for membrane area) ---
AREA_UM2 = MEMBRANE_SIZE[0] * MEMBRANE_SIZE[1]  # 100 um2
POPULATIONS = {
    'PIP2':      int(100.0 * AREA_UM2),   # 10000 — substrate pool
    'PIP3':      int(0.4  * AREA_UM2),     # 40    — LOW state (wave nucleation)
    'PI3K':      int(5.0  * AREA_UM2),     # 500   — kinase
    'PTEN_T1':   int(20.0 * AREA_UM2),     # 2000  — phosphatase (high activity)
    'PTEN_T2':   int(5.0  * AREA_UM2),     # 500   — phosphatase (low activity)
    'PHAkt':     50,                        # biosensor (low expression, passive)
    'PHAkt_PIP3': 0,                        # start unbound
}

# --- Diffusion coefficients (um2/s) ---
# D_PIP3/D_PTEN = 3.3 ratio required for fire-diffuse-fire waves
DIFFUSION = {
    'PIP2':       0.10,   # Arai 2010
    'PIP3':       0.10,   # Arai 2010
    'PI3K':       0.10,   # kinase, intermediate
    'PTEN_T1':    0.03,   # Matsuoka 2018 (slow membrane-bound)
    'PTEN_T2':    0.03,   # Matsuoka 2018
    'PHAkt':      0.08,   # peripheral membrane protein
    'PHAkt_PIP3': 0.06,   # biosensor-lipid complex
}

# --- Catalytic rates (s-1) ---
CATALYSIS = {
    'PI3K':    0.15,   # PIP2 -> PIP3 (with pip3_activation feedback)
    'PTEN_T1': 0.50,   # PIP3 -> PIP2 (high activity form)
    'PTEN_T2': 0.05,   # PIP3 -> PIP2 (low activity form)
}
CONTACT_RADIUS_NM = 150.0  # electrostatic capture radius (Saha 2014)

# --- Compartment exchange rates (s-1) ---
EXCHANGE = {
    'PI3K':    {'k_on_factor': 80.0, 'k_off_factor': 80.0},  # fast (adiabatic)
    'PTEN_T1': {'k_on': 0.05, 'k_off': 0.02},                # slow (wave timescale)
    'PHAkt':   {'k_on': 5.0, 'k_off': 0.3, 'cytosol_pool': 250},
}

# --- Feedback rules ---
FEEDBACK = {
    'pip3_inhibition':      {'K_half': 5.0,  'n_hill': 3.0, 'basal_fraction': 0.10},
    'pip3_activation':      {'K_half': 3.0,  'n_hill': 3.0, 'basal_fraction': 0.05},
    'pip3_activation_pten': {'K_half': 15.0},
}
GRID_RESOLUTION = 1.0  # um (for local density feedback calculation)

# --- Biosensor binding ---
PHAKT_KOFF = 3.0    # s-1 (tau=333ms, f_bound~42%)
PHAKT_KON  = 0.05   # um2/s (diffusion-limited)

# --- Bleach reduction ---
BLEACH_REDUCTION = 500.0  # divide bleach rates by this factor

print(f'PIP3-PTEN wave system configured:')
print(f'  Membrane: {MEMBRANE_SIZE[0]}x{MEMBRANE_SIZE[1]} um ({AREA_UM2} um2)')
print(f'  Total particles: {sum(POPULATIONS.values())}')
print(f'  D ratio PIP3/PTEN: {DIFFUSION["PIP3"]/DIFFUSION["PTEN_T1"]:.1f}x')

## 3. Apply Configuration and Run

In [ ]:
config = load_config(YAML_CONFIG)

# --- Apply simulation parameters ---
config.bd_timestep = BD_TIMESTEP
config.scan.n_frames = N_FRAMES
config.scan.frame_interval = N_BD_PER_FRAME * BD_TIMESTEP
config.scan.dwell_time = DWELL_TIME
config.membrane.size = MEMBRANE_SIZE
config.scan.pixels_x = PIXELS
config.scan.pixels_y = PIXELS
config.scan.pixel_size = PIXEL_SIZE_NM

# --- Apply populations ---
for sp_name, count in POPULATIONS.items():
    if sp_name in config.initial_populations:
        config.initial_populations[sp_name] = count

# --- Apply diffusion ---
for sp in config.species:
    if sp.name in DIFFUSION:
        sp.diffusion_coefficient = DIFFUSION[sp.name]

# --- Apply catalytic rates ---
for cat in config.catalytic_reactions:
    for key, rate in CATALYSIS.items():
        if key in cat.name:
            cat.k_cat = rate
            cat.contact_radius = CONTACT_RADIUS_NM
    if 'PI3K' in cat.name:
        cat.feedback_rule = 'pip3_activation'

# --- Apply compartment exchanges ---
for exch in config.compartment_exchanges:
    for key, params in EXCHANGE.items():
        if key in exch.name:
            if 'k_on' in params:
                exch.k_on = params['k_on']
            if 'k_off' in params:
                exch.k_off = params['k_off']
            if 'k_on_factor' in params:
                exch.k_on *= params['k_on_factor']
            if 'k_off_factor' in params:
                exch.k_off *= params['k_off_factor']
            if 'cytosol_pool' in params:
                exch.cytosol_pool = params['cytosol_pool']
    exch.max_recruits_per_step = 20

# --- Apply feedback rules ---
for fb in config.feedback_rules:
    if fb.name in FEEDBACK:
        params = FEEDBACK[fb.name]
        if 'K_half' in params:
            fb.K_half = params['K_half']
        if 'n_hill' in params:
            fb.n_hill = params['n_hill']
        if 'basal_fraction' in params:
            fb.basal_fraction = params['basal_fraction']
    fb.grid_resolution = GRID_RESOLUTION

# --- Apply biosensor kinetics ---
for rxn in config.reactions:
    if 'PHAkt' in rxn.name:
        rxn.koff = PHAKT_KOFF
        rxn.kon = PHAKT_KON

# --- Apply bleach reduction ---
for fluor in config.fluorophores:
    for state, transitions in fluor.thermal_rates.items():
        if 'bleached' in transitions:
            transitions['bleached'] /= BLEACH_REDUCTION

# --- Apply unimolecular conversions ---
for conv in config.unimolecular_conversions:
    if conv.name == 'T1_to_T2':
        conv.rate = 0.35

# --- Raft centering for larger domain ---
for reg in config.diffusion_field.regions:
    if reg.get('shape') == 'circle':
        reg['center'] = [MEMBRANE_SIZE[0]/2, MEMBRANE_SIZE[1]/2]
        reg['radius'] = 2.0

config.validate()
print(f'Config validated: {config.name}')
print(f'  {N_FRAMES} frames x {config.scan.frame_interval}s = {N_FRAMES * config.scan.frame_interval}s total biology')

In [ ]:
# Run all replicas
sim_results = []
t0 = time.perf_counter()

for r in range(N_REPLICAS):
    cfg = copy.deepcopy(config)
    if hasattr(cfg, 'random_seed'):
        cfg.random_seed = 2000 + r * 13
    
    pipe = SimulationPipeline(cfg, record_subframe=True)
    result = pipe.run()
    sim_results.append(result)
    
    complexes = [f.ground_truth.n_complexes for f in result.frames]
    print(f'  Replica {r:2d}: {result.total_photons_detected:6d} photons, '
          f'{result.total_wall_time_s:.0f}s wall, '
          f'complexes: {complexes[0]}->{complexes[-1]}')

wall_sim = time.perf_counter() - t0
print(f'\nTotal: {wall_sim:.0f}s ({wall_sim/60:.1f} min) for {N_REPLICAS} replicas')

## 4. Video Generation

In [ ]:
os.makedirs(VIDEO_DIR, exist_ok=True)

t0_vid = time.perf_counter()
video_paths = generate_all_replica_videos(
    sim_results, output_dir=VIDEO_DIR,
    prefix='pip3_pten', fmt=VIDEO_FORMAT,
    tau_d_ns=TAU_D_NS, fps=VIDEO_FPS, dpi=VIDEO_DPI)
wall_vid = time.perf_counter() - t0_vid

print(f'\n{len(video_paths)} replicas x 3 = {len(video_paths)*3} videos')
print(f'Time: {wall_vid:.1f}s')

In [ ]:
# Display replica 0
print('=== Replica 0: BD Particles ===')
display(Image(filename=video_paths[0]['bd_particles']))
print('\n=== Replica 0: Fluorescence Intensity ===')
display(Image(filename=video_paths[0]['intensity']))
print('\n=== Replica 0: FLIM Lifetime ===')
display(Image(filename=video_paths[0]['flim']))

## 5. Transfer Entropy v2

In [ ]:
trials_dict = sim_results_to_trials_dict(sim_results, tau_d_ns=TAU_D_NS)

print('Series extracted:')
for key, arr in trials_dict.items():
    print(f'  {key:15s}: shape={arr.shape}, '
          f'range=[{arr.min():.4f}, {arr.max():.4f}]')

In [ ]:
causal_pairs = [
    CausalPairSpecV2(
        source_key='n_complexes', target_key='tau_mean',
        candidate_lags=TE_CANDIDATE_LAGS, n_bins=TE_N_BINS,
        n_surrogates=N_SURROGATES, lag_criterion=TE_LAG_CRITERION,
        detrend_method=TE_DETREND,
    ),
    CausalPairSpecV2(
        source_key='n_bleached', target_key='intensity',
        candidate_lags=TE_CANDIDATE_LAGS, n_bins=TE_N_BINS,
        n_surrogates=N_SURROGATES, lag_criterion=TE_LAG_CRITERION,
        detrend_method=TE_DETREND,
    ),
    CausalPairSpecV2(
        source_key='msd', target_key='n_complexes',
        candidate_lags=TE_CANDIDATE_LAGS, n_bins=TE_N_BINS,
        n_surrogates=N_SURROGATES, lag_criterion=TE_LAG_CRITERION,
        detrend_method=TE_DETREND,
    ),
    CausalPairSpecV2(
        source_key='density', target_key='E_mean',
        candidate_lags=TE_CANDIDATE_LAGS, n_bins=TE_N_BINS,
        n_surrogates=N_SURROGATES, lag_criterion=TE_LAG_CRITERION,
        detrend_method=TE_DETREND,
    ),
    CausalPairSpecV2(
        source_key='E_mean', target_key='n_bleached',
        candidate_lags=TE_CANDIDATE_LAGS, n_bins=TE_N_BINS,
        n_surrogates=N_SURROGATES, lag_criterion=TE_LAG_CRITERION,
        detrend_method=TE_DETREND,
    ),
]

t0_te = time.perf_counter()
te_results = compute_causal_flim_generic_v2(trials_dict, causal_pairs)
wall_te = time.perf_counter() - t0_te

print(f'\n{"Pair":<35s} {"TE(X->Y)":>10s} {"TE(Y->X)":>10s} {"p-value":>10s} {"Sig?":>6s} {"Lag":>5s}')
print('-' * 76)
for res in te_results:
    spec = res.spec
    te = res.te_result
    pair_name = f'{spec.source_key} -> {spec.target_key}'
    sig = 'YES' if te.sig_xy and te.sig_xy.significant else 'no'
    p = te.sig_xy.p_value if te.sig_xy else float('nan')
    print(f'{pair_name:<35s} {te.te_xy_bits:>10.4f} {te.te_yx_bits:>10.4f} '
          f'{p:>10.4f} {sig:>6s} {te.lag:>5d}')
print(f'\nTE computation: {wall_te:.1f}s')

## 6. Convergence & Save

In [ ]:
import matplotlib.pyplot as plt

x_all = trials_dict['n_complexes']
y_all = trials_dict['tau_mean']

trial_counts = list(range(2, N_REPLICAS + 1))
te_values = []
for n_t in trial_counts:
    xd = np.array([discretize_uniform(t, TE_N_BINS) for t in x_all[:n_t]])
    yd = np.array([discretize_uniform(t, TE_N_BINS) for t in y_all[:n_t]])
    te_values.append(ensemble_conditional_te_discrete(xd, yd, lag=1))

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(trial_counts, te_values, 'o-', color='#4CAF50', linewidth=2, markersize=6)
ax.set_xlabel('Number of trials (replicas)', fontsize=11)
ax.set_ylabel('TE (bits)', fontsize=11)
ax.set_title('Convergence: TE(n_complexes -> tau_mean) vs n_trials', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/convergence_te.png', dpi=150)
plt.show()

In [ ]:
# Save all results
output_data = {
    'system': 'PIP3-PTEN',
    'n_replicas': N_REPLICAS,
    'n_frames': N_FRAMES,
    'membrane_size': MEMBRANE_SIZE,
    'total_particles': sum(POPULATIONS.values()),
    'wall_time_sim_s': wall_sim,
    'wall_time_vid_s': wall_vid,
    'wall_time_te_s': wall_te,
    'pairs': [],
    'convergence': [{'n_trials': n, 'te_bits': v} for n, v in zip(trial_counts, te_values)],
    'series_stats': {k: {'mean': float(v.mean()), 'std': float(v.std()),
                         'min': float(v.min()), 'max': float(v.max())}
                     for k, v in trials_dict.items()},
    'populations': POPULATIONS,
    'diffusion': DIFFUSION,
    'catalysis': CATALYSIS,
    'feedback': {k: {kk: vv for kk, vv in v.items()} for k, v in FEEDBACK.items()},
}

for res in te_results:
    te = res.te_result
    output_data['pairs'].append({
        'source': res.spec.source_key,
        'target': res.spec.target_key,
        'te_xy_bits': te.te_xy_bits,
        'te_yx_bits': te.te_yx_bits,
        'p_xy': te.sig_xy.p_value if te.sig_xy else None,
        'significant_xy': bool(te.sig_xy and te.sig_xy.significant),
        'lag': te.lag,
        'notes': list(te.notes),
    })

results_file = f'{OUTPUT_DIR}/PIP3_PTEN_full_results.json'
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(results_file, 'w') as f:
    json.dump(output_data, f, indent=2, default=str)

np.savez(f'{OUTPUT_DIR}/PIP3_PTEN_trials.npz', **trials_dict)

print(f'Results: {results_file}')
print(f'Trials: {OUTPUT_DIR}/PIP3_PTEN_trials.npz')
print(f'Videos: {VIDEO_DIR}/ ({len(video_paths)*3} files)')
print(f'\n=== PIP3-PTEN ANALYSIS COMPLETE ===')